# Decision Tree - Heart Disease Prediction Dataset

## 1. Konu Anlatımı: Decision Tree (Karar Ağacı) Nedir?

**Decision Tree (Karar Ağacı)**, sınıflandırma ve regresyon problemlerinde kullanılan denetimli bir makine öğrenmesi algoritmasıdır. Ağaç yapısındaki bu model, veriyi belirli kurallara göre dallara ayırarak karar verir.

### Temel Kavramlar:
- **Kök Düğüm (Root Node):** Ağacın en tepesindeki, tüm veriyi içeren düğümdür.
- **Dallar (Branches):** Karar kurallarını temsil eder.
- **İç Düğüm (Internal Node):** Bir özelliğe göre test yapılan ve dallara ayrılan noktalardır.
- **Yaprak Düğüm (Leaf Node):** Nihai kararın (sınıf etiketi) bulunduğu en uç noktalardır.

### Nasıl Çalışır?
1. Tüm veri kök düğümde başlar.
2. Her düğümde, veriyi en iyi şekilde ayıran özellik (feature) seçilir. Bu seçim genellikle **Gini Impurity (Saflık Derecesi)** veya **Entropy (Bilgi Kazancı)** kriterlerine göre yapılır.
3. Seçilen özellik için bir eşik değeri belirlenir ve veri iki alt gruba ayrılır.
4. Bu işlem, belirli bir durma kriterine (örneğin maksimum derinliğe ulaşılması veya düğümdeki örnek sayısının minimum sınırın altına düşmesi) kadar tekrarlanır.

---
## 2. GEREKLİ KÜTÜPHANELERİN YÜKLENMESİ

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Kutuphaneler basariyla yuklendi.")

---
## 3. VERİ SETİNİN YÜKLENMESİ VE ÖN İŞLEME

Bu çalışmada, UCI Machine Learning Repository'de bulunan popüler **Heart Disease (Kalp Hastalığı)** veri setini kullanacağız. Amacımız, hastaların klinik özelliklerine bakarak kalp hastalığı olup olmadığını (`target`: 1 veya 0) tahmin etmektir.

In [ ]:
# Veri setini dogrudan GitHub raw linki uzerinden cekiyoruz
url = "https://raw.githubusercontent.com/murderandmayhem/heart-disease-prediction/master/heart.csv"
df = pd.read_csv(url)

# Verinin ilk 5 satirini inceleyelim
print("Veri Seti Ilk 5 Satir:")
display(df.head())

print(f"\nVeri Seti Boyutu: {df.shape}")

### Eksik Değer Kontrolü

In [ ]:
print("Eksik Deger Sayisi:")
print(df.isnull().sum())

### Hedef Değişken (Target) Dağılımı

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, palette='Set2')
plt.title('Kalp Hastaligi Durumu Dagilimi (0 = Saglikli, 1 = Hasta)')
plt.xlabel('Durum')
plt.ylabel('Kisi Sayisi')
plt.show()

---
## 4. VERİNİN EĞİTİM VE TEST OLARAK AYRILMASI

Modelin performansını objektif değerlendirebilmek için veriyi %70 Eğitim (Train) ve %30 Test olarak ikiye ayırıyoruz.

In [ ]:
# Ozellikler (X) ve Hedef Degisken (y) ayrimi
X = df.drop('target', axis=1)
y = df['target']

# Egitim ve test setlerine bolme
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

print(f"Egitim seti boyutu: {X_train.shape}")
print(f"Test seti boyutu: {X_test.shape}")

---
## 5. DECISION TREE MODELİNİN OLUŞTURULMASI VE EĞİTİLMESİ

İlk aşamada modelin aşırı öğrenmesini (overfitting) kontrol etmek adına `max_depth=4` parametresiyle sınırlandırılmış, Gini kriterine göre çalışan bir Karar Ağacı modeli eğitiyoruz.

In [ ]:
# Model nesnesinin olusturulmasi
clf_tree = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)

# Modelin egitilmesi
clf_tree.fit(X_train, y_train)

print("Karar Agaci modeli basariyla egitildi.")

---
## 6. MODEL PERFORMANSININ DEĞERLENDİRİLMESİ

Eğitilen modelin doğruluğunu hem eğitim hem de test verileri üzerinde inceleyelim.

In [ ]:
# Tahminlerin yapilmasi
y_pred_train = clf_tree.predict(X_train)
y_pred_test = clf_tree.predict(X_test)

# Dogruluk skorlari
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print(f"Egitim Verisi Dogruluk Skoru (Training Accuracy): {train_acc:.4f}")
print(f"Test Verisi Dogruluk Skoru (Testing Accuracy)  : {test_acc:.4f}")

print("\n--- Siniflandirma Raporu (Test Verisi) ---")
print(classification_report(y_test, y_pred_test))

### Karmaşıklık Matrisi (Confusion Matrix)

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Saglikli', 'Hasta'], yticklabels=['Saglikli', 'Hasta'])
plt.title('Confusion Matrix - Kalp Hastaligi Tahmini')
plt.xlabel('Tahmin Edilen Sinif')
plt.ylabel('Gercek Sinif')
plt.show()

---
## 7. ÖZELLİK ÖNEM DERECELERİ (FEATURE IMPORTANCE)

Karar ağacının kararları verirken hangi klinik özelliklere (örneğin göğüs ağrısı tipi, maksimum kalp atış hızı vb.) daha çok güvendiğini grafik üzerinde görelim.

In [ ]:
importances = clf_tree.feature_importances_
features = X.columns

df_importance = pd.DataFrame({'Ozellik': features, 'Onem Derecesi': importances}).sort_values(by='Onem Derecesi', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Onem Derecesi', y='Ozellik', data=df_importance, palette='viridis')
plt.title('Karar Agacina Gore Ozelliklerin Onem Siralamasi')
plt.xlabel('Onem Derecesi')
plt.ylabel('Klinik Ozellikler')
plt.show()

---
## 8. KARAR AĞACININ GÖRSELLEŞTİRİLMESİ

Modelin kök düğümden başlayarak dallara nasıl ayrıldığını şema üzerinde görelim.

In [ ]:
plt.figure(figsize=(20, 12))
plot_tree(clf_tree, feature_names=X.columns, class_names=['Saglikli', 'Hasta'], filled=True, rounded=True, fontsize=10)
plt.title('Egitilen Karar Agacinin Dallanma Yapisi (Max Depth = 4)')
plt.show()

---
## 9. MODEL KARMAŞIKLIĞI VE OVERFITTING ANALİZİ

Ağacın maksimum derinlik parametresinin (`max_depth`) model başarısı üzerindeki etkisini analiz edelim. Derinlik arttıkça modelin ezberleyip ezberlemediğini grafik yardımıyla saptayacağız.

In [ ]:
training_accuracy = []
testing_accuracy = []
depths = range(1, 16)

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    
    training_accuracy.append(accuracy_score(y_train, model.predict(X_train)))
    testing_accuracy.append(accuracy_score(y_test, model.predict(X_test)))

# Degisimi çizelim
plt.figure(figsize=(10, 6))
plt.plot(depths, training_accuracy, label="Egitim Dogrulugu (Train Accuracy)", marker='o', color='blue')
plt.plot(depths, testing_accuracy, label="Test Dogrulugu (Test Accuracy)", marker='s', color='orange')
plt.ylabel("Dogruluk (Accuracy)")
plt.xlabel("Maksimum Derinlik (max_depth)")
plt.title("Karar Agaclarinda Derinlige Gore Overfitting Analizi")
plt.xticks(depths)
plt.grid(True)
plt.legend()
plt.show()

## Özet ve Sonuç

- **Decision Tree** algoritmasini bu kez kalp hastaligi risk tahmini veri seti uzerinde basariyla uyguladik.
- Veriyi homojen bir dagilimla egitim/test olarak ayirdik (%70 / %30).
- Gini saflik kriteriyle siniflandirma modelini calistirdik.
- Hastaligin tespitinde en onemli ayirt edici klinik ozelliklerin hangileri oldugunu Feature Importance grafigiyle netlestirdik.
- `max_depth` analizi yaparak modelin hangi derinlikten sonra egitim verisini ezberlemeye (overfitting) basladigini, test dogrulugunun nerede optimize oldugunu gozlemledik.